# 11 Map CUSIP to PERMNO

目标：按CUSIP8+rdate_month精确映射PERMNO并排除一对多键及非完全重复证券键。

- 输入：10持仓；CRSP月度
- 输出：wficn_rdate_permno_holdings.parquet及三类映射审计
- 主键：wficn+rdate+permno
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：10
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import duckdb
from src.atomic_write import duckdb_copy_csv_atomic, duckdb_copy_parquet_atomic
from src.runtime_paths import output_path, upstream_path

def esc(path): return str(path).replace("'","''")
hold_path=upstream_path(ROOT,RUN_MODE,"10_map_s12_holdings_to_wficn","data/interim/10_s12_wficn/wficn_report_holdings_selected.parquet")
limit="" if not SMOKE else " LIMIT 500000"
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET preserve_insertion_order=false")
con.execute(f"""CREATE TEMP TABLE stock AS
 SELECT DISTINCT
   lpad(substr(trim(CUSIP),1,8),8,'0') cusip8,
   date_trunc('month',try_cast(MthCalDt AS DATE))::DATE rdate_month,
   try_cast(PERMNO AS BIGINT) permno
 FROM (
   SELECT PERMNO,CUSIP,MthCalDt
   FROM read_csv_auto('{esc(PATHS["crsp_monthly"])}',header=true,all_varchar=true)
   {limit}
 )
 WHERE try_cast(PERMNO AS BIGINT) IS NOT NULL
   AND try_cast(MthCalDt AS DATE) IS NOT NULL""")
con.execute("""CREATE TEMP TABLE multi_keys AS
 SELECT cusip8,rdate_month,count(DISTINCT permno) n_permno
 FROM stock GROUP BY cusip8,rdate_month HAVING count(DISTINCT permno)>1""")
con.execute("""CREATE TEMP VIEW usable AS
 SELECT s.* FROM stock s
 ANTI JOIN multi_keys m USING(cusip8,rdate_month)""")
con.execute(f"""CREATE TEMP VIEW holdings AS
 SELECT *,date_trunc('month',rdate)::DATE rdate_month
 FROM read_parquet('{esc(hold_path)}')""")
con.execute("""CREATE TEMP VIEW mapped_all AS
 SELECT h.*,u.permno
 FROM holdings h LEFT JOIN usable u USING(cusip8,rdate_month)""")
con.execute("""CREATE TEMP TABLE mapped AS
 SELECT DISTINCT * FROM mapped_all WHERE permno IS NOT NULL""")
con.execute("""CREATE TEMP TABLE conflict_keys AS
 SELECT wficn,rdate,permno,count(*) n
 FROM mapped GROUP BY wficn,rdate,permno HAVING count(*)>1""")
con.execute("""CREATE TEMP VIEW clean AS
 SELECT m.* FROM mapped m
 ANTI JOIN conflict_keys c USING(wficn,rdate,permno)""")
out=output_path(ROOT,RUN_MODE,"11_map_cusip_to_permno","data/interim/11_security_map/wficn_rdate_permno_holdings.parquet")
audit1=output_path(ROOT,RUN_MODE,"11_map_cusip_to_permno","data/audit/11_cusip_month_multi_permno.csv")
audit2=output_path(ROOT,RUN_MODE,"11_map_cusip_to_permno","data/audit/11_unmatched_cusip.csv")
audit3=output_path(ROOT,RUN_MODE,"11_map_cusip_to_permno","data/audit/11_wficn_rdate_permno_conflicts.csv")
duckdb_copy_parquet_atomic(con,"SELECT * FROM clean",out)
duckdb_copy_csv_atomic(con,"""SELECT s.*,m.n_permno,'cusip_month_multiple_permno' reason_code
 FROM stock s JOIN multi_keys m USING(cusip8,rdate_month)""",audit1)
duckdb_copy_csv_atomic(con,"""SELECT *,'no_exact_cusip_month_permno' reason_code
 FROM mapped_all WHERE permno IS NULL""",audit2)
duckdb_copy_csv_atomic(con,"""SELECT m.*,c.n,'nonunique_wficn_rdate_permno' reason_code
 FROM mapped m JOIN conflict_keys c USING(wficn,rdate,permno)""",audit3)
hold_count=int(con.execute("SELECT count(*) FROM holdings").fetchone()[0])
stock_count=int(con.execute("SELECT count(*) FROM stock").fetchone()[0])
clean_count=int(con.execute("SELECT count(*) FROM clean").fetchone()[0])
unmatched_count=int(con.execute("SELECT count(*) FROM mapped_all WHERE permno IS NULL").fetchone()[0])
conflict_count=int(con.execute("SELECT coalesce(sum(n),0) FROM conflict_keys").fetchone()[0])
duplicate_count=int(con.execute("""SELECT coalesce(sum(n-1),0) FROM (
 SELECT count(*) n FROM clean GROUP BY wficn,rdate,permno HAVING count(*)>1)""").fetchone()[0])
assert duplicate_count==0
con.close()
INPUTS_USED=["wficn_report_holdings_selected.parquet","monthly_stock_file.csv"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,audit1,audit2,audit3]]
INPUT_COUNTS={"holdings":hold_count,"stock_map":stock_count}; OUTPUT_COUNTS={"mapped_holdings":clean_count}; PRIMARY_KEY=["wficn","rdate","permno"]
DUPLICATE_KEY_COUNT=duplicate_count; EXCLUDED_ROW_COUNT=unmatched_count+conflict_count; EXCEPTION_FILE=str(audit1.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["One-to-many CUSIP8-month keys were excluded.","Full mapping used external-memory DuckDB joins and atomic outputs."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "11_map_cusip_to_permno", stage="11", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
